<a href="https://colab.research.google.com/github/emredeveloper/Transformers--General-AI/blob/main/Mixture_of_Experts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    """Simple feed-forward network for a single expert"""
    def __init__(self, input_dim, hidden_dim):
        super(Expert, self).__init__()
        self.ffn = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        return self.ffn(x)

class Router(nn.Module):
    """Router: decides which expert to activate."""
    def __init__(self, input_dim, num_experts):
        super(Router, self).__init__()
        self.gate = nn.Linear(input_dim, num_experts)

    def forward(self, x):
        # Compute probabilities for each expert
        return F.softmax(self.gate(x), dim=-1)

class MoELayer(nn.Module):
    """Mixture of Experts layer"""
    def __init__(self, input_dim, hidden_dim, num_experts, top_k=2):
        super(MoELayer, self).__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.experts = nn.ModuleList([Expert(input_dim, hidden_dim) for _ in range(num_experts)])
        self.router = Router(input_dim, num_experts)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()  # Extract dimensions
        x_flat = x.view(-1, x.size(-1))  # Merge batch and sequence dimensions

        # Select experts via the router
        route_weights = self.router(x_flat)
        topk_weights, topk_indices = torch.topk(route_weights, self.top_k, dim=-1)

        # Combine the outputs of the selected experts
        outputs = torch.zeros_like(x_flat)
        for i in range(self.top_k):
            weight = topk_weights[:, i].unsqueeze(-1)
            expert_idx = topk_indices[:, i]
            outputs += weight * torch.cat(
                [self.experts[expert](x_flat[j].unsqueeze(0)) for j, expert in enumerate(expert_idx)], dim=0
            )

        # Restore the original shape
        outputs = outputs.view(batch_size, seq_len, -1)
        return outputs

class MoETransformer(nn.Module):
    """Simple Transformer with MoE"""
    def __init__(self, input_dim, hidden_dim, num_heads, num_experts, top_k):
        super(MoETransformer, self).__init__()
        self.attention = nn.MultiheadAttention(embed_dim=input_dim, num_heads=num_heads, batch_first=True)
        self.moe_layer = MoELayer(input_dim, hidden_dim, num_experts, top_k)
        self.norm1 = nn.LayerNorm(input_dim)
        self.norm2 = nn.LayerNorm(input_dim)

    def forward(self, x):
        # Multi-head attention
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_output)

        # Mixture of Experts layer
        moe_output = self.moe_layer(x)
        x = self.norm2(x + moe_output)

        return x

# Example usage
input_dim = 128
hidden_dim = 256
num_heads = 4
num_experts = 3
top_k = 2
seq_len = 10
batch_size = 5

# Build the model
model = MoETransformer(input_dim, hidden_dim, num_heads, num_experts, top_k)

# Random input data
x = torch.rand(batch_size, seq_len, input_dim)

# Output
output = model(x)
print("Output shape:", output.shape)